In [1]:
import os
import torch
import pandas as pd

from tqdm import tqdm

from stock_gpt import StockGPT, LinearModel, NaiveModel
from dataloader_builder import build_dataloaders
from setup import StockGPT_cfg, LinearModel_cfg, NaiveModel_cfg
from setup import path_data_preprocessor, PATH_RESULTS
from model_training import model_setup, train_model_cuda

from model_training import train_model_cuda, evaluate_model, evaluate_best_model
from model_analysis import test_model, print_loss_analysis, process_losses, format_num, process_result, store_result

In [2]:
cuda = True if torch.cuda.is_available() else False

print("PyTorch:", torch.__version__)
print("CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PyTorch: 2.13.0+cu132
CUDA build: 13.2
CUDA available: True
GPU: NVIDIA GeForce RTX 4070 Laptop GPU


## MODEL TRAINING ---------------------------

In [3]:
torch.manual_seed(1234)
dls, train_norms = build_dataloaders(path_data_preprocessor)

Building DataLoaders...


In [4]:
optimizer_data = [torch.optim.AdamW, 0.0004, 0.1]
scaler_data = [torch.amp.GradScaler, "cuda"]

max_epochs = 15
eval_bs = 1000

stockGPT, stockGPT_params, opt1, sca1, sch1 = model_setup(StockGPT, StockGPT_cfg, train_norms, device,
                                                *optimizer_data, *scaler_data)
linearModel, linearModel_params, opt2, sca2, sch2 = model_setup(LinearModel, LinearModel_cfg, train_norms, device, 
                                                      *optimizer_data, *scaler_data)
naiveModel = NaiveModel(NaiveModel_cfg, train_norms)
naiveModel.to(device)

model_train_losses, model_val_losses = train_model_cuda(stockGPT, device, opt1, sca1, sch1, max_epochs, 
                                                        dls["train"], dls["val"], eval_bs)
linear_train_losses, linear_val_losses = train_model_cuda(linearModel, device, opt2, sca2, sch2, max_epochs,
                                                        dls["train"], dls["val"], eval_bs)


Input Norm: torch.Size([13])|torch.Size([13])
Target Norm: torch.Size([4])|torch.Size([4])
3181824
5632
Continuing from previous checkpoint...


|          | 0.0% (00:00) Setting up...                                                                   

Epoch 16:

Finished
Continuing from previous checkpoint...


|          | 0.0% (00:00) Setting up...                                                                   

Epoch 16:

Finished


## Model Analysis -------------------------

In [ ]:
#* REUSES OBJETCS FROM TRAINING
analysis_steps = min(eval_bs, len(dls["train"])) + min(eval_bs, len(dls["val"])) + min(eval_bs, len(dls["test"]))
analysis_pbar = tqdm(total=3*analysis_steps, desc=f"Evaluating the best model parameters...".ljust(80),
                bar_format="|{bar}| {percentage:3.1f}% ({elapsed}) {desc}", position=0, leave=False)

#* Reevaluates models by their best parameters on train and val dataloaders
naive_losses = evaluate_model(dls["train"], dls["val"], naiveModel, device, eval_bs, analysis_pbar)
linear_losses = evaluate_best_model(linearModel, device, opt2, sca2, sch2, dls["train"], dls["val"], eval_bs, analysis_pbar, True)
gpt_losses = evaluate_best_model(stockGPT, device, opt1, sca1, sch1, dls["train"], dls["val"], eval_bs, analysis_pbar, True) 

#* Final evaluation on unseen test dataloader
naive_test_losses = test_model(dls["test"], naiveModel, device, eval_bs, analysis_pbar)
linear_test_losses = test_model(dls["test"], linearModel, device, eval_bs, analysis_pbar)
gpt_test_losses = test_model(dls["test"], stockGPT, device, eval_bs, analysis_pbar)


|█▋        | 16.7% (00:14) Evaluating model on training data... (999/1000) [1000/6003]:                   

In [ ]:
for key, features in [("NLL", StockGPT_cfg["target_features"]),
                      ("STD", [f"{feature}_std" for feature in StockGPT_cfg["target_features"]]),
                      ("MAE", StockGPT_cfg["target_features"]),
                      ("PMAE", StockGPT_cfg["target_features"])]:
    print_loss_analysis(process_losses(gpt_losses + gpt_test_losses +
                                       linear_losses + linear_test_losses +
                                       naive_losses + naive_test_losses, key), 
                                       [stockGPT.cfg["name"], linearModel.cfg["name"], naiveModel.cfg["name"]],
                                       [format_num(stockGPT_params), format_num(linearModel_params), "0"], 
                                       features, key)


--------------------------------------------------------------------------------------------------------------

NLL

--------------------------------------------------------------------------------------------------------------

                    o        h        l        c        
StockGPT-B5: 3.2M
    Training:       -3.8202  -3.7044  -3.7534  -3.7110    >  -3.7473
    Validation:     -3.7725  -3.6830  -3.7142  -3.6882    >  -3.7145
    Testing:        -3.8035  -3.7147  -3.7493  -3.7241    >  -3.7479
    
LinearModel-B5: 5.6K
    Training:       74805.34381.9492   0.5174   184858.9531  >  64916.6914
    Validation:     0.6307   1.8388   0.5838   0.6493     >  0.9256
    Testing:        0.6350   1.9628   0.5856   0.6539     >  0.9593
    
NaiveModel-B5: 0
    Training:       0.9190   0.9190   0.9190   0.9190     >  0.9190
    Validation:     0.9190   0.9190   0.9190   0.9190     >  0.9190
    Testing:        0.9190   0.9190   0.9190   0.9190     >  0.9190
    

-------------------

In [ ]:
store_result(PATH_RESULTS, process_result(stockGPT, max_epochs, gpt_losses, gpt_test_losses))
store_result(PATH_RESULTS, process_result(linearModel, max_epochs, linear_losses, linear_test_losses))
store_result(PATH_RESULTS, process_result(naiveModel, max_epochs, naive_losses, naive_test_losses))

print(pd.read_parquet(PATH_RESULTS))

{'model': 'StockGPT-B5', 'bar_width': '5', 'epoch': 15, 'train': {'NLL': tensor([-3.8202, -3.7044, -3.7534, -3.7110], device='cuda:0'), 'STD': tensor([0.0078, 0.0085, 0.0084, 0.0086], device='cuda:0'), 'MAE': tensor([0.0036, 0.0043, 0.0037, 0.0040], device='cuda:0'), 'PMAE': tensor([0.4140, 0.4847, 0.4362, 0.4508], device='cuda:0')}, 'val': {'NLL': tensor([-3.7725, -3.6830, -3.7142, -3.6882], device='cuda:0'), 'STD': tensor([0.0086, 0.0092, 0.0091, 0.0093], device='cuda:0'), 'MAE': tensor([0.0042, 0.0047, 0.0043, 0.0045], device='cuda:0'), 'PMAE': tensor([0.3988, 0.4477, 0.4207, 0.4240], device='cuda:0')}, 'test': ({'NLL': tensor([-3.8035, -3.7147, -3.7493, -3.7241], device='cuda:0'), 'STD': tensor([0.0084, 0.0090, 0.0088, 0.0091], device='cuda:0'), 'MAE': tensor([0.0042, 0.0047, 0.0042, 0.0044], device='cuda:0'), 'PMAE': tensor([0.3962, 0.4435, 0.4178, 0.4170], device='cuda:0')},)}


|██████████| 100.0% (01:59) Evaluating model on testing data... (387/388) [6003/6003]: 

PermissionError: [WinError 5] Failed to open local file 'results'. Detail: [Windows error 5] Access is denied.
